# Silver — clean orders

**Layer:** validated, typed, analysis-ready.

- Drop rows with missing `order_id`, qty <= 0, or unit_price <= 0
- Compute `revenue = qty * unit_price`


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("env", "dev")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not catalog or not schema:
    raise ValueError("Job parameters catalog and schema are required")

bronze_table = f"{catalog}.{schema}.example_orders_bronze"
silver_table = f"{catalog}.{schema}.example_orders_silver"


In [ ]:
bronze_df = spark.table(bronze_table)

silver_df = (
    bronze_df.filter(F.trim(F.col("order_id")) != "")
    .filter(F.col("qty") > 0)
    .filter(F.col("unit_price") > 0)
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("revenue", F.col("qty") * F.col("unit_price"))
    .withColumn("order_date", F.to_date("order_date"))
)

silver_df.write.mode("overwrite").saveAsTable(silver_table)
print(f"Wrote {silver_df.count()} clean rows to {silver_table} (env={env})")
silver_df.show(5, truncate=False)
